# 技能4 · Day 2 上机：价值创造机制 + 定价策略

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **statsmodels** 对真实AI API定价数据拟合 OLS 回归，量化"什么驱动了AI产品定价"
2. 用 **numpy-financial** 计算 NPV/IRR/payback，基于真实训练成本（DeepSeek V3 $5.576M）评估定价策略财务可行性
3. 用 **scipy.stats** 估计价格弹性及置信区间，理解小样本下定价决策的不确定性
4. 对比四种定价策略（成本加成/价值定价/渗透/撇脂）的利润曲线，找最优价格点
5. 用**天道推演**框架做定价策略的竞争反应沙盘推演（2026前沿）

## 真实库与真实数据
- **statsmodels**：OLS回归量化定价驱动因素
- **numpy-financial**：NPV/IRR/payback财务建模
- **scipy.stats**：价格弹性估计与置信区间
- **真实AI API定价**：OpenAI/Anthropic/Google/DeepSeek/Mistral官方定价页（非模拟数据）
- **真实训练成本**：DeepSeek V3技术报告公开披露$5.576M

> 详见 data/README.md


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 需要 statsmodels, numpy-financial, scipy, pandas。均为本地库，不需要API Key。


In [ ]:
# !pip install statsmodels numpy-financial scipy pandas -q


## 1. 真实AI API定价数据

**数据来源**：各AI提供商官方定价页（2025-2026），每个数字可追溯验证。

| 提供商 | 官方定价页 |
|--------|-----------|
| OpenAI | https://openai.com/api/pricing/ |
| Anthropic | https://www.anthropic.com/pricing |
| Google | https://ai.google.dev/pricing |
| DeepSeek | https://api-docs.deepseek.com/quick_start/pricing |
| Mistral | https://mistral.ai/products/la-plateforme#pricing |

**价值创造机制分类**（基于独立教材Day2三维度框架）：
- **efficiency**（效率提升）：低价位、快速响应、自动化替代（GPT-4o-mini, Haiku, Flash, DeepSeek-V3）
- **experience**（体验重塑）：中价位、多模态、个性化体验（GPT-4o, Sonnet, Gemini Pro）
- **innovation**（模式创新）：高价位、前沿推理能力、新能力范式（o1, o3, Opus, DeepSeek-R1）

**核心分析任务**：
1. OLS回归：什么驱动了AI产品定价？（output_price ~ context_window + mechanism + reasoning + provider）
2. NPV/IRR：基于真实训练成本和API定价，评估AI产品投资可行性
3. 价格弹性：用真实价格点估计弹性，计算置信区间
4. 定价策略对比：成本加成 vs 价值定价 vs 渗透 vs 撇脂
5. 天道推演：定价策略的竞争反应沙盘推演


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import numpy_financial as npf
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 真实AI API定价数据（来自各提供商官方定价页，2025-2026）
# Sources:
#   OpenAI:    https://openai.com/api/pricing/
#   Anthropic: https://www.anthropic.com/pricing
#   Google:    https://ai.google.dev/pricing
#   DeepSeek:  https://api-docs.deepseek.com/quick_start/pricing
#   Mistral:   https://mistral.ai/products/la-plateforme#pricing
# ============================================================
data = {
    'model': [
        'GPT-4o', 'GPT-4o-mini', 'o1', 'o1-mini', 'o3', 'o3-mini',
        'Claude-3-Opus', 'Claude-3.5-Sonnet', 'Claude-3-Haiku', 'Claude-3.5-Haiku',
        'Gemini-1.5-Pro', 'Gemini-1.5-Flash',
        'DeepSeek-V3', 'DeepSeek-R1',
        'Mistral-Large-2', 'Mistral-Small',
    ],
    'provider': [
        'OpenAI','OpenAI','OpenAI','OpenAI','OpenAI','OpenAI',
        'Anthropic','Anthropic','Anthropic','Anthropic',
        'Google','Google',
        'DeepSeek','DeepSeek',
        'Mistral','Mistral',
    ],
    'input_price': [
        2.50, 0.15, 15.00, 1.10, 10.00, 1.10,
        15.00, 3.00, 0.25, 0.80,
        1.25, 0.075,
        0.14, 0.55,
        2.00, 0.20,
    ],
    'output_price': [
        10.00, 0.60, 60.00, 4.40, 40.00, 4.40,
        75.00, 15.00, 1.25, 4.00,
        5.00, 0.30,
        0.28, 2.19,
        6.00, 0.60,
    ],
    'context_window': [
        128000, 128000, 200000, 128000, 200000, 200000,
        200000, 200000, 200000, 200000,
        2000000, 1000000,
        64000, 64000,
        128000, 32000,
    ],
    'value_mechanism': [
        'experience','efficiency','innovation','innovation','innovation','innovation',
        'innovation','experience','efficiency','efficiency',
        'experience','efficiency',
        'efficiency','innovation',
        'experience','efficiency',
    ],
    'has_reasoning': [
        0,0,1,1,1,1,
        0,0,0,0,
        0,0,
        0,1,
        0,0,
    ],
    'has_vision': [
        1,1,0,0,1,0,
        1,1,1,1,
        1,1,
        0,0,
        0,0,
    ],
}
df = pd.DataFrame(data)
df['blended_price'] = (df['input_price'] + 3 * df['output_price']) / 4

print(f"数据形状: {df.shape}")
print(f"提供商: {list(df['provider'].unique())}")
print(f"价值机制: {list(df['value_mechanism'].unique())}")
print(f"\noutput_price范围: ${df['output_price'].min():.3f} ~ ${df['output_price'].max():.2f} /1M tokens")
print(f"跨数量级: {df['output_price'].max() / df['output_price'].min():.0f}x")


## 2. TODO 1：真实AI API定价数据加载与探索

**任务**：加载真实AI API定价数据，探索数据结构和分布。

**提示**：
- 数据已内嵌在代码中（来自各提供商官方定价页）
- `df.describe()` 查看数值列统计
- `df.groupby('provider')['output_price'].mean()` 对比各提供商定价
- `df.groupby('value_mechanism')['output_price'].mean()` 对比价值创造机制的定价差异

**理论连接**：真实AI API定价数据跨5个数量级（$0.075 ~ $75/1M tokens），不服从正态分布。理解这种跨数量级分布是AI定价分析的基础。

**营销映射**：efficiency类模型（如GPT-4o-mini）适合AI文案生成API按token计费；experience类（如GPT-4o）适合营销Agent按席位订阅；innovation类（如o1）适合AI定价优化引擎按价值分成。


In [ ]:
# TODO 1：真实AI API定价数据加载与探索
# 提示：df.describe() 查看统计
#   df.groupby('provider')['output_price'].mean() 对比各提供商
#   df.groupby('value_mechanism')['output_price'].mean() 对比价值机制
# 要求：打印描述统计、按提供商分组、按价值机制分组、按推理能力分组

# ===== 你的代码 =====
desc_stats = None           # 用 df.describe()
provider_compare = None     # 用 df.groupby('provider')['output_price'].agg(['mean','min','max'])
mechanism_compare = None    # 用 df.groupby('value_mechanism')['output_price'].agg(['mean','min','max'])
reasoning_compare = None    # 用 df.groupby('has_reasoning')['output_price'].mean()
# ====================

print("=== 描述统计 ===")
print(desc_stats)
print("\n=== 按提供商分组 ===")
print(provider_compare)
print("\n=== 按价值创造机制分组 ===")
print(mechanism_compare)
print("\n=== 按推理能力分组 (0=无, 1=有) ===")
print(reasoning_compare)


## 3. TODO 2：OLS回归 -- 什么驱动了AI产品定价？

**任务**：用 statsmodels 拟合 OLS 回归 log(output_price) ~ log(context_window) + value_mechanism + has_reasoning + provider，解读结果。

**提示**：
- 价格跨数量级，需取log：`df['log_output_price'] = np.log(df['output_price'])`
- 创建虚拟变量：efficiency为基线，experience和innovation为dummy
- provider也需虚拟变量：DeepSeek为基线
- `X = sm.add_constant(X)` 添加截距项
- `model = sm.OLS(y, X).fit()` 拟合模型
- `model.summary()` 查看完整结果

**要求**：
1. 拟合OLS模型，打印summary
2. 提取R²、各系数及其p值
3. 解读：哪些变量显著？价值创造机制（efficiency/experience/innovation）的系数差异说明了什么？

**营销解读**：如果mech_innovation系数显著为正，说明"模式创新"型AI产品（如推理模型o1）可以定高价--这支持"价值定价"而非"成本加成"的定价策略。


In [ ]:
# TODO 2：OLS回归 -- 什么驱动了AI产品定价？
# 提示：1) df['log_output_price'] = np.log(df['output_price'])
#       2) df['log_context'] = np.log(df['context_window'])
#       3) 创建虚拟变量：mech_experience, mech_innovation (baseline=efficiency)
#       4) 创建provider虚拟变量 (baseline=DeepSeek)
#       5) X = sm.add_constant(df[X_cols])
#       6) model_ols = sm.OLS(df['log_output_price'], X).fit()
# 要求：拟合OLS，打印summary，提取R²/系数/p值

# ===== 你的代码 =====
df['log_output_price'] = None  # 用 np.log(df['output_price'])
df['log_context'] = None       # 用 np.log(df['context_window'])
df['mech_experience'] = None   # (df['value_mechanism']=='experience').astype(int)
df['mech_innovation'] = None   # (df['value_mechanism']=='innovation').astype(int)
df['prov_openai'] = None       # (df['provider']=='OpenAI').astype(int)
df['prov_anthropic'] = None    # (df['provider']=='Anthropic').astype(int)
df['prov_google'] = None       # (df['provider']=='Google').astype(int)
df['prov_mistral'] = None      # (df['provider']=='Mistral').astype(int)

X_cols = None                  # 列表：['log_context','mech_experience','mech_innovation','has_reasoning','prov_openai','prov_anthropic','prov_google','prov_mistral']
X_ols = None                   # sm.add_constant(df[X_cols])
model_ols = None               # sm.OLS(df['log_output_price'], X_ols).fit()
rsquared = None                # model_ols.rsquared
coefs = None                   # model_ols.params
pvalues = None                 # model_ols.pvalues
# ====================

print("=== OLS回归结果 ===")
print(model_ols.summary())
print(f"\nR² = {rsquared:.6f}")
print(f"调整R² = {model_ols.rsquared_adj:.6f}")
print(f"F p-value = {model_ols.f_pvalue:.4f}")
print("\n=== 系数与p值 ===")
for name in coefs.index:
    sig = "***" if pvalues[name] < 0.01 else ("**" if pvalues[name] < 0.05 else ("*" if pvalues[name] < 0.1 else ""))
    print(f"  {name}: coef={coefs[name]:.4f}, p={pvalues[name]:.4f} {sig}")


## 4. TODO 3：NPV/IRR/payback -- AI产品财务可行性分析

**任务**：用 numpy-financial 计算AI产品的 NPV/IRR/payback，基于真实训练成本和API定价。

**背景**：
- **初始投资**：$5,576,000（DeepSeek V3公开披露的训练成本）
- **推理成本**：$0.30/1M tokens（vLLM优化后的估计）
- **基准月活量**：20,000M tokens（200亿token/月）
- **年增长率**：50%（1.5x）
- **贴现率**：10%
- **投资期限**：5年

**四种定价策略**：
1. **成本加成**（50%加成）：$0.45/1M tokens，高量
2. **价值定价**（Claude Sonnet水平）：$3/$15 per 1M（input/output），中等量
3. **渗透定价**（DeepSeek V3水平）：$0.14/$0.28 per 1M，超高量
4. **撇脂定价**（Claude Opus水平）：$15/$75 per 1M，低量

**提示**：
- `npf.npv(rate, cashflows)` 计算NPV
- `npf.irr(cashflows)` 计算IRR
- payback period需要手动计算累积现金流
- 现金流：Year 0 = -initial_investment, Year 1-5 = monthly_profit * 12 * growth^(yr-1)

**营销解读**：NPV告诉你"这个AI产品值不值得投"；IRR告诉你"投资的年化回报率"。如果渗透定价的NPV为负，说明DeepSeek的低价策略在财务上不可持续--除非推理成本远低于$0.30/1M。


In [ ]:
# TODO 3：NPV/IRR/payback -- AI产品财务可行性分析
# 提示：1) initial_investment = 5_576_000 (DeepSeek V3真实训练成本)
#       2) inference_cost = 0.30 ($/1M tokens)
#       3) 四种策略有不同的price和volume_multiplier
#       4) cashflows = [-initial_investment, year1_profit, ..., year5_profit]
#       5) npf.npv(rate, cashflows), npf.irr(cashflows)
# 要求：对四种策略计算月度利润、5年现金流、NPV、IRR、payback

initial_investment = 5_576_000  # DeepSeek V3真实训练成本 ($)
inference_cost_per_1m = 0.30    # $/1M tokens (vLLM优化后)
base_monthly_volume = 20000     # 20000M tokens/月 (基准量)
annual_growth = 1.5             # 年增长50%
discount_rate = 0.10            # 10%贴现率
years = 5
input_output_ratio = 4.0        # 典型input:output = 4:1

strategies = {
    'cost_plus': {'input_price': 0.45, 'output_price': 0.45, 'volume_mult': 8.0},
    'value_based': {'input_price': 3.00, 'output_price': 15.00, 'volume_mult': 1.0},
    'penetration': {'input_price': 0.14, 'output_price': 0.28, 'volume_mult': 15.0},
    'skimming': {'input_price': 15.00, 'output_price': 75.00, 'volume_mult': 0.1},
}

# ===== 你的代码 =====
results = {}  # 字典：{策略名: {'npv':, 'irr':, 'payback':, 'monthly_profit':, 'cashflows':}}

for name, s in strategies.items():
    monthly_input_vol = None    # base_monthly_volume * s['volume_mult']
    monthly_output_vol = None   # monthly_input_vol / input_output_ratio
    monthly_revenue = None      # monthly_input_vol * s['input_price'] + monthly_output_vol * s['output_price']
    monthly_cost = None         # (monthly_input_vol + monthly_output_vol) * inference_cost_per_1m
    monthly_profit = None       # monthly_revenue - monthly_cost

    cashflows = None            # [-initial_investment] + [monthly_profit*12*growth^(yr-1) for yr in 1..5]
    npv_val = None              # npf.npv(discount_rate, cashflows)
    irr_val = None              # npf.irr(cashflows)

    # payback (手动计算累积现金流)
    payback_val = None          # 累积现金流达到initial_investment的年份

    results[name] = {
        'monthly_profit': monthly_profit,
        'npv': npv_val,
        'irr': irr_val,
        'payback': payback_val,
        'cashflows': cashflows,
    }
# ====================

print(f"初始投资: ${initial_investment:,} (DeepSeek V3训练成本)")
print(f"推理成本: ${inference_cost_per_1m}/1M tokens\n")
for name, r in results.items():
    print(f"{name}:")
    print(f"  月利润: ${r['monthly_profit']:,.2f}")
    print(f"  NPV(10%,5yr): ${r['npv']:,.2f}")
    irr_str = f"{r['irr']*100:.2f}%" if r['irr'] is not None and not np.isnan(r['irr']) else "N/A"
    print(f"  IRR: {irr_str}")
    pb_str = f"{r['payback']:.2f}年" if r['payback'] is not None else f">{years}年"
    print(f"  Payback: {pb_str}\n")


## 5. TODO 4：价格弹性估计与置信区间

**任务**：用 scipy.stats 估计AI API的价格弹性，计算置信区间。

**背景**：价格弹性（Price Elasticity）衡量"价格变化1%时需求变化百分之几"。弹性 < -1 为弹性需求（降价增收），弹性 > -1 为非弹性需求（涨价增收）。

**方法**：用真实AI API价格点和对应的市场需求量（基于市场定位建模），做log-log回归估计弹性。

**提示**：
- `stats.linregress(log_p, log_q)` 做简单线性回归，斜率即弹性
- `r_value**2` 为R²
- `std_err` 为斜率标准误
- 95% CI: `slope ± t.ppf(0.975, df) * std_err`
- `stats.t.ppf(0.975, n-2)` 获取t临界值

**要求**：
1. 用真实价格点和建模需求量做log-log回归
2. 计算弹性、R²、p值、标准误
3. 计算弹性的95%置信区间
4. 解读：AI API是弹性需求还是非弹性需求？这对定价策略有什么启示？

**理论连接**：小样本（n=10）下，频率派的弹性点估计不稳定。贝叶斯方法（PyMC）可通过先验分布提供正则化，给出弹性的后验分布--这是TODO6天道推演的理论基础。


In [ ]:
# TODO 4：价格弹性估计与置信区间
# 提示：1) 用真实API价格点和建模需求量
#       2) stats.linregress(log_p, log_q) 做log-log回归
#       3) 斜率 = 弹性
#       4) 95% CI: slope ± t.ppf(0.975, df) * std_err
# 要求：计算弹性、R²、p值、标准误、95% CI

# 真实AI API价格点 ($/1M tokens, 从官方定价页采集)
price_points = np.array([0.075, 0.14, 0.28, 0.60, 1.25, 3.00, 10.00, 15.00, 40.00, 75.00])
# 建模需求量（基于市场定位：低价模型高量，高价模型低量）
demand_points = np.array([100, 95, 80, 60, 45, 30, 15, 10, 3, 1])

# ===== 你的代码 =====
log_p = None       # np.log(price_points)
log_q = None       # np.log(demand_points)
slope = None       # stats.linregress(log_p, log_q) 的第一个返回值
intercept = None   # 第二个返回值
r_value = None     # 第三个返回值
p_value_elas = None # 第四个返回值
std_err_elas = None # 第五个返回值
elasticity = None   # = slope

n_elas = None       # len(price_points)
df_freedom = None   # n_elas - 2
t_crit = None       # stats.t.ppf(0.975, df_freedom)
ci_lower = None     # slope - t_crit * std_err_elas
ci_upper = None     # slope + t_crit * std_err_elas
# ====================

print("=== 价格弹性估计 (log-log回归) ===")
print(f"  价格点: {price_points}")
print(f"  需求点: {demand_points}")
print(f"  弹性 = {elasticity:.4f}")
print(f"  R² = {r_value**2:.4f}")
print(f"  p-value = {p_value_elas:.6f}")
print(f"  std_err = {std_err_elas:.4f}")
print(f"  95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  t-critical (df={df_freedom}): {t_crit:.4f}")
print(f"\n  解读: |弹性|={abs(elasticity):.4f} {'< 1 (非弹性需求)' if abs(elasticity) < 1 else '>= 1 (弹性需求)'}")


## 6. TODO 5：四种定价策略利润对比

**任务**：用需求曲线模型对比四种定价策略的利润，找最优价格点。

**需求模型**：`Q(P) = Q0 * (P_ref / P)^|elasticity|`
- Q0 = 20,000M tokens（基准月量）
- P_ref = $3.00（参考价格，Claude Sonnet水平）
- elasticity = TODO4估计的弹性

**四种策略价格点**：
- 成本加成：$0.45/1M
- 价值定价：$3.00/1M
- 渗透定价：$0.14/1M
- 撇脂定价：$15.00/1M

**提示**：
- `prices = np.linspace(0.10, 80, 1000)` 生成价格区间
- `volumes = Q0 * (P_ref / prices) ** abs_elas` 计算各价格下的需求量
- `profits = (prices - C) * volumes` 计算利润
- `np.argmax(profits)` 找最优价格

**要求**：
1. 生成价格-需求-利润曲线
2. 找到理论最优价格
3. 对比四种策略在需求曲线模型下的利润
4. 解读：为什么撇脂策略在非弹性需求下利润更高？这与TODO3的NPV结果一致吗？

**营销解读**：当需求非弹性时（|elasticity| < 1），高价策略利润更高--这是OpenAI/Anthropic能定高价的经济学基础。但TODO3的NPV分析显示value_based策略NPV更高，因为TODO3考虑了增长和贴现。


In [ ]:
# TODO 5：四种定价策略利润对比
# 提示：1) 需求模型: Q(P) = Q0 * (P_ref / P)^|elasticity|
#       2) profit = (P - C) * Q(P)
#       3) np.linspace 生成价格区间
#       4) np.argmax 找最优价格
# 要求：生成利润曲线，找最优价格，对比四种策略

C = 0.30                    # 推理成本 $/1M tokens
Q0 = 20000                  # 基准月量 (1M tokens)
P_ref = 3.00                # 参考价格
abs_elas = abs(elasticity)  # 弹性绝对值

# ===== 你的代码 =====
prices = None        # np.linspace(0.10, 80, 1000)
volumes = None       # Q0 * (P_ref / prices) ** abs_elas
profits = None       # (prices - C) * volumes

opt_idx = None       # np.argmax(profits)
opt_price = None     # prices[opt_idx]
opt_profit = None    # profits[opt_idx]
opt_volume = None    # volumes[opt_idx]

# 四种策略在需求曲线模型下的利润
strategy_prices = {'cost_plus': 0.45, 'value_based': 3.00, 'penetration': 0.14, 'skimming': 15.00}
strategy_results = {}  # {name: {'price':, 'volume':, 'profit':}}
for sname, sp in strategy_prices.items():
    sv = None      # Q0 * (P_ref / sp) ** abs_elas
    sp_profit = None  # (sp - C) * sv
    strategy_results[sname] = {'price': sp, 'volume': sv, 'profit': sp_profit}
# ====================

print(f"=== 需求曲线模型 ===")
print(f"  成本: ${C}/1M tokens")
print(f"  基准量: {Q0}M tokens/月 (at P_ref=${P_ref})")
print(f"  弹性: {abs_elas:.4f}")
print(f"\n  理论最优价格: ${opt_price:.2f}/1M tokens")
print(f"  最优月量: {opt_volume:,.0f}M tokens")
print(f"  最优月利润: ${opt_profit:,.2f}")

print(f"\n{'策略':<15} {'价格':>8} {'月量':>12} {'月利润':>15}")
print("-" * 55)
for sname, sr in strategy_results.items():
    print(f"{sname:<15} ${sr['price']:>7.2f} {sr['volume']:>11,.0f}M ${sr['profit']:>13,.2f}")

print(f"\n  解读: 需求{'非弹性' if abs_elas < 1 else '弹性'}(|e|={abs_elas:.4f})，")
print(f"  {'高价策略利润更高（撇脂最优）' if abs_elas < 1 else '低价策略利润更高（渗透最优）'}")


## 7. TODO 6：天道推演 -- 定价策略竞争反应沙盘（2026前沿）

**任务**：用天道推演框架模拟定价策略的竞争反应路径，做蒙特卡洛仿真。

**天道推演框架**：
```
输入：我方定价策略 + 竞争格局
处理：
  1. 局势拆解 -> 识别竞争者、替代品
  2. 因果建模 -> 定价->市场份额->竞争反应
  3. 沙盘展开 -> 每个策略生成3层推演树
     - immediate: 竞品即时反应
     - near (3-6月): 市场格局调整
     - far (1-2年): 行业生态演化
  4. 概率注入 -> 各反应路径的概率分布
输出：期望利润 + 风险预警
```

**三种策略的竞争反应概率**：

| 策略 | 竞品跟降 | 竞品不变 | 竞品降价反击 | 竞品退出 |
|------|---------|---------|------------|---------|
| 渗透 | 55% | 25% | 5%(价格战) | 15% |
| 价值定价 | 30% | 50% | 15% | 5% |
| 撇脂 | 20% | 60% | 15% | 5% |

**提示**：
- `np.random.choice(reactions, p=probs)` 按概率采样反应
- 10,000次蒙特卡洛仿真
- 3层推演：immediate (6个月) -> near (6个月) -> far (12个月)
- 价格战far期利润衰减50%；竞品退出far期利润增长20%

**要求**：
1. 对三种策略各做10,000次仿真
2. 计算期望2年总利润、标准差、5th/95th分位数
3. 计算 P(利润>0)
4. 输出各竞品反应的概率分布和对应利润
5. 解读：哪种策略的天道推演期望最优？风险最大的是哪种？

**营销映射**：天道推演本质是"计算化的沙盘推演"。在AI营销SaaS定价决策中，如果你采用渗透定价，竞品（如OpenAI）跟降的概率是多少？价格战升级的概率是多少？这些可以用历史定价战数据校准概率。


In [ ]:
# TODO 6：天道推演 -- 定价策略竞争反应沙盘（2026前沿）
# 提示：1) 定义三种策略的竞品反应概率
#       2) np.random.choice 按概率采样
#       3) 10,000次蒙特卡洛仿真
#       4) 3层推演：immediate(6月) -> near(6月) -> far(12月)
# 要求：计算期望2年利润、标准差、5th/95th分位、P(利润>0)

np.random.seed(42)
n_simulations = 10000
Q0_sim = 20000  # 基准月量
C_sim = 0.30    # 推理成本

scenarios = {
    'penetration': {
        'desc': '渗透定价 ($0.14/1M)',
        'reactions': {
            'competitor_match':  {'prob': 0.55, 'share': 0.30, 'price': 0.14, 'vol_mult': 3.0},
            'competitor_ignore': {'prob': 0.25, 'share': 0.60, 'price': 0.14, 'vol_mult': 5.0},
            'competitor_exit':   {'prob': 0.15, 'share': 0.80, 'price': 0.14, 'vol_mult': 8.0},
            'price_war':         {'prob': 0.05, 'share': 0.20, 'price': 0.07, 'vol_mult': 2.0},
        }
    },
    'value_based': {
        'desc': '价值定价 ($3/1M)',
        'reactions': {
            'competitor_match':    {'prob': 0.30, 'share': 0.35, 'price': 3.00, 'vol_mult': 1.0},
            'competitor_ignore':   {'prob': 0.50, 'share': 0.45, 'price': 3.00, 'vol_mult': 1.2},
            'competitor_undercut': {'prob': 0.15, 'share': 0.25, 'price': 2.00, 'vol_mult': 1.5},
            'competitor_exit':     {'prob': 0.05, 'share': 0.60, 'price': 3.00, 'vol_mult': 2.0},
        }
    },
    'skimming': {
        'desc': '撇脂定价 ($15/1M)',
        'reactions': {
            'competitor_ignore':   {'prob': 0.60, 'share': 0.15, 'price': 15.00, 'vol_mult': 0.1},
            'competitor_match':    {'prob': 0.20, 'share': 0.10, 'price': 15.00, 'vol_mult': 0.08},
            'competitor_undercut': {'prob': 0.15, 'share': 0.08, 'price': 10.00, 'vol_mult': 0.12},
            'competitor_exit':     {'prob': 0.05, 'share': 0.25, 'price': 15.00, 'vol_mult': 0.15},
        }
    },
}

# ===== 你的代码 =====
sim_results = {}  # {策略名: DataFrame of simulation results}

for strat_name, strat in scenarios.items():
    reaction_names = list(strat['reactions'].keys())
    probs = [strat['reactions'][r]['prob'] for r in reaction_names]

    sim_data = []
    for _ in range(n_simulations):
        reaction = None      # np.random.choice(reaction_names, p=probs)
        r = strat['reactions'][reaction]

        # 3层推演
        base_volume = None   # Q0_sim * r['vol_mult'] * r['share']
        immediate_profit = None  # (r['price'] - C_sim) * base_volume
        near_profit = None       # immediate_profit * 0.80

        # far层根据反应类型调整
        if reaction == 'price_war':
            far_profit = None    # immediate_profit * 0.50
        elif reaction == 'competitor_exit':
            far_profit = None    # immediate_profit * 1.20
        else:
            far_profit = None    # immediate_profit * 0.90

        total_2yr = immediate_profit * 6 + near_profit * 6 + far_profit * 12
        sim_data.append({'reaction': reaction, 'immediate': immediate_profit,
                         'near': near_profit, 'far': far_profit, 'total_2yr': total_2yr})

    sim_results[strat_name] = pd.DataFrame(sim_data)
# ====================

for strat_name, strat in scenarios.items():
    res = sim_results[strat_name]
    print(f"\n{strat_name}: {strat['desc']}")
    print(f"  期望2年利润: ${res['total_2yr'].mean():,.2f}")
    print(f"  标准差: ${res['total_2yr'].std():,.2f}")
    print(f"  5th分位: ${res['total_2yr'].quantile(0.05):,.2f}")
    print(f"  95th分位: ${res['total_2yr'].quantile(0.95):,.2f}")
    print(f"  P(利润>0): {(res['total_2yr']>0).mean():.4f}")
    print(f"  反应分布:")
    for r in strat['reactions']:
        count = (res['reaction']==r).sum()
        avg = res[res['reaction']==r]['total_2yr'].mean()
        print(f"    {r}: {count/n_simulations*100:.1f}% -> 平均2年利润: ${avg:,.2f}")


## 8. 反思与前沿

### 反思问题
1. OLS回归中哪些变量对AI产品定价有显著影响？价值创造机制（efficiency/experience/innovation）的系数差异说明了什么？
2. R²=0.859但Adj R²=0.698，为什么差距大？这说明了什么？（提示：样本量n=16，自变量8个）
3. 渗透定价的NPV为什么是负的？DeepSeek如何在真实中维持低价？（提示：推理成本可能远低于$0.30/1M）
4. 价格弹性估计的95% CI有多宽？这对定价决策意味着什么？
5. 天道推演中，哪种策略的期望利润最高？风险（标准差）最大的是哪种？
6. 如果用贝叶斯方法替代频率派估计弹性，结果会有什么不同？

### 2026前沿：贝叶斯定价 + 推理成本定价 + 天道推演

**贝叶斯定价（PyMC）**：传统弹性估计给点估计+置信区间，贝叶斯方法给后验分布。小样本下通过先验分布提供正则化。关键词"贝叶斯"连接2026因果推断前沿。

**推理成本定价**：AI产品按token推理成本定价。vLLM（PagedAttention）、投机解码（Speculative Decoding）持续降低推理成本，为降价创造空间。DeepSeek以$0.14/1M input（约为GPT-4o的1/18）迫使行业重新思考定价基准。

**天道推演动态定价**：定价不是静态决策而是动态博弈。天道推演系统性预判竞争反应：如果我降价，竞品跟降的概率是多少？价格战升级的路径如何展开？这比单纯计算NPV更有战略价值。

### 从定价分析到商业模式设计

今天的定价分析是Day1商业模式画布的核心组件，也是Day3 Agent经济的基础。在Agent经济中，定价模型从seat-based转向outcome-based--当Agent替代人类执行任务时，按"人头"收费失去意义，按结果付费是唯一合理的定价模型。

> "定价不是数学问题，而是博弈问题。天道推演帮你看清博弈的棋盘。"
